# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Run each expensive algorithm in a separate, interruptible process.
# The 30-second inference budget excludes imports and KB construction.
import os
import subprocess
import sys
import textwrap

worker = textwrap.dedent(r"""
    import json, sys, time
    from sudoku_solver import atom, build_general_kb
    from logic_ import pl_resolution, tt_entails, associate, prop_symbols
    raw = json.load(open('puzzles.json'))
    givens = {tuple(map(int, k.split('_'))): v
              for k, v in raw['puzzles'][0]['givens'].items()}
    kb = build_general_kb(raw['n'], raw['box_h'], raw['box_w'], givens)
    # Query an initially empty cell; the target value does not use solution.
    r, c = next((r, c) for r in range(1, raw['n'] + 1)
                for c in range(1, raw['n'] + 1) if (r, c) not in givens)
    query = atom('Is', r, c, 1)
    sentence = associate('&', kb.clauses)
    print(json.dumps({'ready': True, 'query': str(query),
                      'clauses': len(kb.clauses),
                      'symbols': len(prop_symbols(sentence))}), flush=True)
    start = time.perf_counter()
    try:
        result = (pl_resolution(kb, query) if sys.argv[1] == 'resolution'
                  else tt_entails(sentence, query))
        record = {'status': 'completed', 'result': result}
    except (MemoryError, RecursionError) as exc:
        record = {'status': type(exc).__name__}
    record['seconds'] = time.perf_counter() - start
    print(json.dumps(record), flush=True)
""")
general_observations = []
for algorithm in ('resolution', 'truth_table'):
    child = subprocess.Popen([sys.executable, '-u', '-c', worker, algorithm],
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                             text=True, env={**os.environ, 'PYTHONHASHSEED': '0'})
    try:
        ready = json.loads(child.stdout.readline())
        start = time.perf_counter()
        try:
            output, error = child.communicate(timeout=30)
            if child.returncode != 0:
                raise RuntimeError(error)
            outcome = json.loads(output.strip())
        except subprocess.TimeoutExpired:
            child.kill()
            child.communicate()
            outcome = {'status': 'timeout', 'seconds': time.perf_counter() - start}
        observation = {'algorithm': algorithm, **ready, **outcome}
        general_observations.append(observation)
        print(observation, flush=True)
    finally:
        if child.poll() is None:
            child.kill()
            child.communicate()

{'algorithm': 'resolution', 'ready': True, 'query': 'Is1_1_1', 'clauses': 10317, 'symbols': 729, 'status': 'timeout', 'seconds': 30.19241483299993}


{'algorithm': 'truth_table', 'ready': True, 'query': 'Is1_1_1', 'clauses': 10317, 'symbols': 729, 'status': 'timeout', 'seconds': 30.00709920801455}


### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


Both attempts use the first pool puzzle and the same initially empty-cell query, with candidate 1 selected without consulting the stored solution. The printed records above contain the actual outcomes. Each child process is stopped after a 30-second inference budget; imports and KB construction are outside that budget. A timeout is an unfinished attempt, **not** a `False` entailment answer. We stop after this representative query rather than attempting hundreds of equally expensive queries. See Q2 for the separate causes of the cost.

**Recorded run (Python 3.10.7, macOS arm64):** both `pl_resolution` and `tt_entails` timed out on `Is1_1_1` with 10,317 clauses and 729 symbols. Observed stop times were approximately 30.19 s and 30.01 s respectively (including small timeout/termination overhead). Neither returned an entailment result. The FC solver subsequently proved this query as part of the verified full-grid solution.


**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
# Both solvers import their implementations from sudoku_solver.py.
solved = solve_full_grid_fc(n, box_h, box_w, givens)
assert solved == puzzle['solution']
print('Forward-chaining solution:')
for r in range(1, n + 1):
    print(' '.join(str(solved[(r, c)]) for c in range(1, n + 1)))

# Demonstrate a real proof for D's tutor mode; no solver is duplicated here.
from sudoku_solver import trace_fc_query
r, c = next(cell for cell in solved if cell not in givens)
query = atom('Is', r, c, solved[(r, c)])
entailed, proof = trace_fc_query(build_definite_kb(n, box_h, box_w, givens), query)
assert entailed
available = set()
for step in proof:
    assert all(p in available for p in step['premises'])
    available.add(step['conclusion'])
assert query in available
print(f'Query {query}: {entailed}; proof contains {len(proof)} facts/deductions.')
print('Final rule:', proof[-1]['rule'])

Forward-chaining solution:
1 3 6 9 8 5 4 7 2
2 9 4 3 1 7 8 6 5
5 8 7 2 6 4 1 3 9
7 5 1 4 9 6 2 8 3
3 6 9 8 7 2 5 1 4
4 2 8 5 3 1 7 9 6
8 1 2 6 4 9 3 5 7
6 4 3 7 5 8 9 2 1
9 7 5 1 2 3 6 4 8


Query Is1_1_1: True; proof contains 299 facts/deductions.
Final rule: ((Not1_1_2 & Not1_1_3 & Not1_1_4 & Not1_1_5 & Not1_1_6 & Not1_1_7 & Not1_1_8 & Not1_1_9) ==> Is1_1_1)


### Verification and reproducible timing

Validate both full-grid results on all five puzzles. Check BC on **all 729 cell/value queries per puzzle**, including all incorrect values. The full FC closure determines those same answers: exactly one `Is` value is derived per cell. Stored solutions are used only in assertions after inference.

The timing experiment uses three fresh calls per algorithm per puzzle, alternates which algorithm runs first, and includes KB construction, indexing, inference, and grid reconstruction. Validation and trace recording are outside the measured intervals. Raw times, median, range, and BC/FC ratio are printed; comparisons apply to these implementations and this machine.

In [7]:
import platform
import statistics

print(f'Python {platform.python_version()}, {platform.system()} {platform.machine()}')
print('Fresh KB per timed solve; 3 repeats; seconds; ratio = BC median / FC median')
benchmark_results = []
for index, item in enumerate(puzzle_pool):
    # Correct and incorrect candidate verification on a fresh definite KB.
    fc_grid = solve_full_grid_fc(n, box_h, box_w, item['givens'])
    assert fc_grid == item['solution']
    definite_kb = build_definite_kb(n, box_h, box_w, item['givens'])
    checked = 0
    for r in range(1, n + 1):
        for c in range(1, n + 1):
            for v in range(1, n + 1):
                answer = pl_bc_entails(definite_kb, atom('Is', r, c, v))
                assert answer == (fc_grid[(r, c)] == v)
                checked += 1
    times = {'FC': [], 'BC': []}
    solvers = {'FC': solve_full_grid_fc, 'BC': solve_full_grid_bc}
    for repeat in range(3):
        order = ('FC', 'BC') if (repeat + index) % 2 == 0 else ('BC', 'FC')
        for algorithm in order:
            start = time.perf_counter()
            result = solvers[algorithm](n, box_h, box_w, item['givens'])
            elapsed = time.perf_counter() - start
            assert result == item['solution']
            times[algorithm].append(elapsed)
    medians = {key: statistics.median(value) for key, value in times.items()}
    benchmark_results.append({'puzzle': index, 'givens': item['given_count'],
                              'times': times, 'medians': medians,
                              'ratio': medians['BC'] / medians['FC']})
    print(f"Puzzle {index}, givens={item['given_count']}: {checked} candidate checks passed")
    for algorithm in ('FC', 'BC'):
        values = times[algorithm]
        print(f'  {algorithm}: raw={[round(t, 4) for t in values]}, '
              f'median={medians[algorithm]:.4f}, range={min(values):.4f}–{max(values):.4f}')
    print(f"  BC/FC={medians['BC'] / medians['FC']:.2f}x", flush=True)
print('All five full-grid solutions and 3,645 BC candidate queries passed.')

Python 3.10.7, Darwin arm64
Fresh KB per timed solve; 3 repeats; seconds; ratio = BC median / FC median


Puzzle 0, givens=30: 729 candidate checks passed
  FC: raw=[0.3781, 0.4145, 0.3909], median=0.3909, range=0.3781–0.4145
  BC: raw=[1.6966, 1.7583, 2.1528], median=1.7583, range=1.6966–2.1528
  BC/FC=4.50x


Puzzle 1, givens=33: 729 candidate checks passed
  FC: raw=[0.3783, 0.3586, 0.3843], median=0.3783, range=0.3586–0.3843
  BC: raw=[1.8239, 1.8237, 1.8334], median=1.8239, range=1.8237–1.8334
  BC/FC=4.82x


Puzzle 2, givens=36: 729 candidate checks passed
  FC: raw=[0.3467, 0.4212, 0.4103], median=0.4103, range=0.3467–0.4212
  BC: raw=[1.7182, 1.8613, 1.6908], median=1.7182, range=1.6908–1.8613
  BC/FC=4.19x


Puzzle 3, givens=39: 729 candidate checks passed
  FC: raw=[0.3837, 0.3543, 0.4183], median=0.3837, range=0.3543–0.4183
  BC: raw=[1.7995, 1.799, 1.7843], median=1.7990, range=1.7843–1.7995
  BC/FC=4.69x


Puzzle 4, givens=42: 729 candidate checks passed
  FC: raw=[0.3763, 0.4181, 0.3478], median=0.3763, range=0.3478–0.4181
  BC: raw=[1.7699, 1.8148, 1.8601], median=1.8148, range=1.7699–1.8601
  BC/FC=4.82x


All five full-grid solutions and 3,645 BC candidate queries passed.


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**

**(a) General KB**

I use `Is(r,c,v)` to mean that cell `(r,c)` contains value `v`. The constraints are:

- **At least one value:** `Is(r,c,1) | ... | Is(r,c,n)`.
- **At most one value:** `~Is(r,c,u) | ~Is(r,c,v)` for every pair of different values.
- **Row, column and box uniqueness:** for two cells sharing a row, column or box, add `~Is(r1,c1,v) | ~Is(r2,c2,v)` for each value.
- **Givens:** add the given values as `Is` facts.

These are CNF clauses stored in `PropKB`.

**(b) Definite KB**

A definite clause has exactly one positive literal, so we cannot directly use the at-least-one disjunction above. Instead, I use `Not(r,c,v)` as a positive atom meaning that candidate `v` has been excluded.

The KB contains three kinds of knowledge:

1. Given values as `Is` facts.
2. Elimination rules: a known value excludes other values in the same cell and the same value from peer cells.
3. Last-candidate rules: if all other values in a cell are excluded, infer the remaining value.

For example, in a 4×4 puzzle:

$$
\mathrm{Not}_{r,c,1}\land \mathrm{Not}_{r,c,2}\land
\mathrm{Not}_{r,c,3}
\Rightarrow \mathrm{Is}_{r,c,4}
$$

This is definite because its clause form has only one positive literal, the conclusion. `Not` is an independent atom, not automatic logical negation. These rules support elimination, but they are not fully equivalent to the general constraints and may leave some puzzles unsolved.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**

I would not use either supplied general-purpose procedure as the default Sudoku solver. Soundness and completeness describe whether an answer is correct and eventually obtainable with sufficient resources; they do not promise a practical runtime or memory footprint.

**Truth-table model checking.** The general encoding has $m=n^3$ Boolean `Is` symbols, hence 729 symbols for a 9×9 grid. Enumerating Boolean assignments has a worst-case cost of $O(2^m L)$, where $L$ is the formula length, and $2^{729}\approx2.82\times10^{219}$. Most assignments violate Sudoku constraints, but the supplied `tt_check_all` evaluates the KB only after assigning every symbol; it does not propagate givens or prune inconsistent partial assignments. Its depth-first traversal does not store all $2^m$ models at once. This implementation still copies symbol lists and model dictionaries along the recursion stack, so auxiliary storage can be $O(m^2)$, in addition to the formula. Exponential time is the principal obstacle.

**Resolution-refutation.** `pl_resolution` adds the negated query, explicitly constructs every pair of current clauses, and resolves each pair. With $C$ clauses, a single round allocates $C(C-1)/2$ pairs, requiring quadratic pair-list space even before storing new resolvents. Resolving a pair also compares literals, and successive rounds can generate many new clauses and revisit old pairs. The number of distinct non-tautological clauses over $m$ atoms can reach $3^m$, so resolution closure has exponential worst-case growth; unrestricted resolution is very different from an optimized SAT solver. For this puzzle, the initial KB has 10,317 clauses (including 30 givens), so adding the negated query already creates 53,225,403 pairs in the first round.

The experiment above attempts one initially empty-cell query using each supplied procedure and records whether it finishes within 30 seconds. The concrete outcomes are reported in the observation notes; a timed-out run establishes only that it exceeded our budget. It does not disprove completeness or establish a universal lower bound for every query.

For the provided puzzles, indexed forward chaining over the elimination/last-candidate Horn rules is practical. With a premise index, processing the Horn closure is linear in the total number of rule-premise occurrences, apart from expression and indexing overhead. It is complete for that Horn KB, **not** a complete Sudoku-solving technique: the Horn encoding may omit consequences of the full Sudoku constraints. For harder puzzles, stronger rules or a suitable constraint/SAT search method would be needed.

### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**

Backward chaining is goal-driven: it starts with the requested proposition and only explores rules that may establish that proposition. Before answering queries, I separate the definite KB into a set of known facts and an index from each rule conclusion to the rules that can produce it. This avoids scanning the complete KB at every recursive call.

```text
BC-ENTAILS(KB, query):
    if the KB index does not yet exist:
        facts <- all clauses with no premises
        rules_by_conclusion <- group all rules by their conclusion
        proven <- facts

    if query has a cached top-level result:
        return that result

    repeat:
        old_proven_count <- size(proven)
        active <- empty set
        failed_this_pass <- empty set

        result <- PROVE(query)
        if result is true:
            cache query as true and return true
        if size(proven) = old_proven_count:
            cache query as false and return false

PROVE(goal):
    if goal is in proven:
        return true
    if goal is in active or failed_this_pass:
        return false

    add goal to active
    for each rule in rules_by_conclusion[goal]:       // OR over rules
        if PROVE(p) is true for every premise p:      // AND over premises
            add goal to proven
            remove goal from active
            return true

    remove goal from active
    add goal to failed_this_pass
    return false
```

The first challenge was correctly combining evidence. Every premise of one rule must be proved, so premises use AND, while any one of several rules may establish the goal, so candidate rules use OR. A known fact is the recursion base case. If no fact or rule proves the goal, the answer is `False`; failure to prove an `Is` atom does not automatically prove a corresponding `Not` atom, because `Not...` is a separate positive symbol in this KB.

The second challenge was termination. Elimination and last-candidate rules can form dependency cycles. The `active` set records only the current recursion path; encountering an active goal rejects that cyclic branch without rejecting other rules for the same goal. `failed_this_pass` prevents repeated expansion of the same unsuccessful subgoal during one traversal.

A subtle issue is that a goal which fails early in a cyclic traversal may become provable after another branch derives a new fact. Therefore I do not permanently cache recursive failures. Instead, the outer loop repeats the search whenever the `proven` set grows, clearing `failed_this_pass` each time. Successful deductions are monotonic and remain in `proven`. Because the KB contains finitely many atoms, the proven set can grow only finitely many times; when a complete pass proves nothing new, a fixed point has been reached and the query is safely returned as `False`. This provides both cycle safety and guaranteed termination.

Finally, indexing rules by conclusion and caching successful deductions avoids repeatedly scanning the roughly twenty thousand definite rules generated for a 9×9 puzzle. Top-level query results are also safe to reuse because they are recorded only after success or after the fixed point has been reached.

### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**

I choose **Naked Pairs**. If two cells in the same row, column or box have only candidates `{a,b}` remaining, they must contain these two values in some order. Therefore, other cells in that unit cannot contain `a` or `b`.

For example, in a 4×4 puzzle, suppose cells `(1,1)` and `(1,2)` have both excluded 1 and 3. Their remaining candidates are `{2,4}`. We can add:

$$
\mathrm{Not}_{1,1,1}\land \mathrm{Not}_{1,1,3}\land
\mathrm{Not}_{1,2,1}\land \mathrm{Not}_{1,2,3}
\Rightarrow \mathrm{Not}_{1,3,2}
$$

A separate rule with the same premises concludes `Not(1,3,4)`. Similar rules apply to the other cells in that row.

Each rule has positive premises and one positive conclusion, so it is a definite clause. The exclusions must be explicitly proved; simply failing to prove a value is not enough.

Adding these rules could eliminate more candidates and help the solver make progress. However, covering all possible pairs would increase the number of rules, memory usage and inference work. It still would not guarantee solving every puzzle. Our current solver does not implement Naked Pairs; the rules above show how it could be added.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**

**(a) When BC can be faster.** Suppose a large Horn KB contains many unrelated rule groups, but a single query depends on a short chain of rules and a few known facts. With an existing conclusion index, backward chaining explores just that query's relevant ancestors and stops after finding one proof. Forward chaining may instead propagate facts through many irrelevant groups before reaching the target. Index construction still costs a scan of the KB; the strongest benefit arises when an index is reused across queries. A given-cell query with an already indexed KB is an especially cheap BC case.

**(b) When BC has no advantage or is slower.** Solving a full Sudoku grid asks many related queries. Elimination and last-candidate rules form a dense network of shared, cyclic dependencies. BC can spend time exploring unsuccessful candidate proofs and revisiting goals across fixed-point passes. An indexed FC pass derives all reachable facts once, and the same closure answers every cell query. For a query whose relevant rules cover most of the KB, the selective-search advantage of BC is small.

Our FC solver calls the supplied `pl_fc_entails` once with an unreachable marker query so that its agenda is exhausted. A premise-indexing observer collects processed facts; it does not replace the inference algorithm. The grid is reconstructed from that closure. Our BC solver tries candidates in ascending order, reusing its conclusion index, proven atoms and top-level query results within each solve. Each timed call starts with a fresh KB, so neither algorithm inherits caches from validation or an earlier run.

The table/output above gives three repeats on each of the five supplied puzzles, including KB construction, indexing, inference and reconstruction. The measured medians and ratios are summarized below after execution. The comparison measures **a shared-closure FC implementation against this cached BC implementation**; it is not a claim that FC always wins. Repeating the library FC call independently for every candidate, especially with its default whole-KB premise scan, would have substantially different costs. With only one puzzle at each given count, these data also cannot establish that more givens always cause faster inference.

**Measured results (seconds, median of three fresh solves):**

| Puzzle index | Givens | FC | BC | BC/FC |
| --- | --- | --- | --- | --- |
| 0 | 30 | 0.3909 | 1.7583 | 4.50 |
| 1 | 33 | 0.3783 | 1.8239 | 4.82 |
| 2 | 36 | 0.4103 | 1.7182 | 4.19 |
| 3 | 39 | 0.3837 | 1.7990 | 4.69 |
| 4 | 42 | 0.3763 | 1.8148 | 4.82 |

FC was approximately 4.19–4.82 times faster in these five whole-grid experiments. On the working puzzle (30 givens), the medians were 0.3909 s for FC and 1.7583 s for BC. This supports the shared-work explanation in (b), while BC still benefits from indexing and cached proofs. These are local measurements, not portable runtime guarantees.


## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

`https://...`

**Integration status:** A/B/C implementations and Q1–Q5 are present, and all existing code cells have been executed. D still needs to complete the Streamlit application and replace the URL placeholder above before final submission.
